In [2]:
import xarray as xr
import torch
import sys
sys.path.append('../src')
from constants import INPT_VARS, EXTRA_VARS, OUT_VARS
import os
import numpy as np

In [238]:
data_dir = "/pscratch/sd/s/suryad/data"
wet_file = "3D_wet.pt"
surface_wet_file = "surface_wet.pt"
data_zarr = "3D_data"
data_means_zarr = "3D_data_means"
data_stds_zarr = "3D_data_stds"
grid_file = "Grid_New.nc"

inputs = INPT_VARS["3D_all"]
extra_in = EXTRA_VARS["3D_all"]
outputs = OUT_VARS["3D_all"]
lag = 1
interval = 1
hist = 0
N_samples = 20
N_val = 10
steps = 4


wet = torch.load(
    os.path.join(data_dir, wet_file)
)
data = xr.open_zarr(
    os.path.join(data_dir, data_zarr)
)
data_mean = xr.open_zarr(
    os.path.join(data_dir, data_means_zarr)
)
data_std = xr.open_zarr(
    os.path.join(data_dir, data_stds_zarr)
)

s_train = lag * hist
e_train = s_train + N_samples * interval
e_test = e_train + interval * N_val
    


### Main

#### data_CNN_Disk

#### old

In [325]:
class old_data_CNN_Disk(torch.utils.data.Dataset):

    def __init__(
        self,
        data,
        inputs_str,
        extra_in_str,
        outputs_str,
        wet,
        data_mean,
        data_std,
        n_samples,
        lag,
        interval,
        ind_start,
        device="cuda",
    ):
        super().__init__()
        self.device = device

        self.size = n_samples
        self.lag = lag
        self.interval = interval
        self.ind_start = ind_start

        self.inputs = data[inputs_str + extra_in_str]
        self.outputs = data[outputs_str]

        self.in_mean = data_mean[inputs_str + extra_in_str]
        self.in_std = data_std[inputs_str + extra_in_str]

        self.out_mean = data_mean[outputs_str]
        self.out_std = data_std[outputs_str]

        self.wet = wet

    def set_device(self, device):
        self.device = device

    def __len__(self):
        # Number of data point we have. Alternatively self.data.shape[0], or self.label.shape[0]
        return self.size

    def __getitem__(self, idx):
        # Return the idx-th data point of the dataset
        # If we have multiple things to return (data point and label), we can return them as tuple
        if type(idx) == list:
            ind_in = [self.ind_start + i * self.interval for i in idx]
            ind_out = [self.ind_start + i * self.interval + self.lag for i in idx]
        elif type(idx) == slice:
            if idx.start == None and idx.stop == None:
                idx = slice(0, self.size, idx.step)
            elif idx.start == None:
                idx = slice(0, idx.stop, idx.step)
            elif idx.stop == None:
                idx = slice(idx.start, self.size, idx.step)

            ind_in = slice(
                self.ind_start + idx.start, self.ind_start + idx.stop * self.interval, self.interval
            )
            ind_out = slice(
                self.ind_start + idx.start + self.lag,
                self.ind_start + idx.stop * self.interval + self.lag,
                self.interval,
            )
        if type(idx) == int:
            ind_in = self.ind_start + idx * self.interval
            ind_out = self.ind_start + idx * self.interval + self.lag

        data_in = self.inputs.isel(time=ind_in)
        data_in = ((data_in - self.in_mean) / self.in_std).fillna(0)
        label = self.outputs.isel(time=ind_out)
        label = ((label - self.out_mean) / self.out_std).fillna(0)

        if type(idx) == int:
            data_in = data_in.to_array().transpose("variable", "y", "x").to_numpy()
            label = label.to_array().transpose("variable", "y", "x").to_numpy()
        else:
            data_in = (
                data_in.to_array().transpose("time", "variable", "y", "x").to_numpy()
            )
            label = label.to_array().transpose("time", "variable", "y", "x").to_numpy()

        items = (torch.from_numpy(data_in).float(), torch.from_numpy(label).float())

        return items

#### new

In [326]:
class data_CNN_Disk(torch.utils.data.Dataset):

    def __init__(
        self,
        data,
        inputs_str,
        extra_in_str,
        outputs_str,
        wet,
        data_mean,
        data_std,
        n_samples,
        lag,
        interval,
        hist,
        ind_start,
        device="cuda",
    ):
        super().__init__()
        self.device = device

        self.size = n_samples
        self.lag = lag
        self.interval = interval
        self.hist = hist
        
        assert self.interval == 1
        assert self.lag == 1
        
        data = data.isel(time=slice(ind_start, None))
        self.inputs = data[inputs_str + extra_in_str]
        self.outputs = data[outputs_str]
        
        total_steps = self.hist + 1
        
        indices = xr.DataArray(
            np.arange(data.time.size),
            dims=["time"],
            coords={"time": data.time},
        )
        
        rolling_indices = indices.rolling(time=len(data.time)-total_steps, center=False).construct("window_dim").astype(int)
        self.rolling_indices = rolling_indices.transpose('window_dim', 'time').isel(time=slice(len(data.time)-total_steps-1, None))
        self.inputs_no_extra = data[inputs_str]
        self.extras = data[extra_in_str]
        self.inputs_no_extra_mean = data_mean[inputs_str]
        self.inputs_no_extra_std = data_std[inputs_str]
        self.extras_mean = data_mean[extra_in_str]
        self.extras_std = data_std[extra_in_str]

        self.in_mean = data_mean[inputs_str + extra_in_str]
        self.in_std = data_std[inputs_str + extra_in_str]

        self.out_mean = data_mean[outputs_str]
        self.out_std = data_std[outputs_str]

        self.wet = wet

    def set_device(self, device):
        self.device = device

    def __len__(self):
        # Number of data point we have. Alternatively self.data.shape[0], or self.label.shape[0]
        return self.rolling_indices.window_dim.size

    def __getitem__(self, idx):
        # Return the idx-th data point of the dataset
        # If we have multiple things to return (data point and label), we can return them as tuple
        if type(idx) == slice:
            if idx.start == None and idx.stop == None:
                idx = slice(0, self.size, idx.step)
            elif idx.start == None:
                idx = slice(0, idx.stop, idx.step)
            elif idx.stop == None:
                idx = slice(idx.start, self.size, idx.step)
        elif type(idx) == int:
            idx = slice(idx, idx + 1, 1)
        
        data_in = self.inputs_no_extra.isel(time=xr.Variable(["window_dim", "time"], self.rolling_indices.isel(window_dim=idx))).isel(time=slice(None, -1))
        data_in = ((data_in - self.inputs_no_extra_mean) / self.inputs_no_extra_std).fillna(0)
        data_in = data_in.to_array().transpose("window_dim", "time", "variable", "y", "x").to_numpy()
        data_in = data_in.reshape((data_in.shape[0], -1, data_in.shape[3], data_in.shape[4]))
        data_in_boundary = self.extras.isel(time=xr.Variable(["window_dim", "time"], self.rolling_indices.isel(window_dim=idx))).isel(time=-2)
        data_in_boundary = ((data_in_boundary - self.extras_mean) / self.extras_std).fillna(0)
        data_in_boundary = data_in_boundary.to_array().transpose("window_dim", "variable", "y", "x").to_numpy()
        data_in = np.concatenate((data_in, data_in_boundary), axis=1)

        label = self.outputs.isel(time=xr.Variable(["window_dim", "time"], self.rolling_indices.isel(window_dim=idx))).isel(time=-1)
        label = ((label - self.out_mean) / self.out_std).fillna(0)
        label = label.to_array().transpose("window_dim", "variable", "y", "x").to_numpy()

        items = (torch.from_numpy(data_in).float(), torch.from_numpy(label).float())

        return items

In [327]:
hist = 0 
val_data = data_CNN_Disk(
    data,
    inputs,
    extra_in,
    outputs,
    wet,
    data_mean,
    data_std,
    N_val,
    lag,
    interval,
    hist,
    e_train,
    device="cuda",
)

old_val_data = old_data_CNN_Disk(
    data,
    inputs,
    extra_in,
    outputs,
    wet,
    data_mean,
    data_std,
    N_val,
    lag,
    interval,
    e_train,
    device="cuda",
)

/global/common/software/m4331/suryad/conda/emulator/lib/python3.10/site-packages/xarray/core/duck_array_ops.py:188: RuntimeWarning: invalid value encountered in cast
  return data.astype(dtype, **kwargs)


In [331]:
(old_val_data[:][1] == val_data[:][1]).all()

tensor(True)

In [304]:
%%time
d0 = val_data[6]
print(d0[0].shape)
print(d0[1].shape)

torch.Size([1, 80, 180, 360])
torch.Size([1, 77, 180, 360])
CPU times: user 3.74 s, sys: 247 ms, total: 3.99 s
Wall time: 3.4 s


In [312]:
%%time
s = slice(6, 10)
d1 = val_data[s]
print(d1[0].shape)
print(d1[1].shape)

torch.Size([4, 157, 180, 360])
torch.Size([4, 77, 180, 360])
CPU times: user 6.18 s, sys: 2.42 s, total: 8.59 s
Wall time: 4.64 s


In [313]:
# Assertion
idx = s
ind_out = slice(
    idx.start + val_data.hist + val_data.lag,
    idx.stop * val_data.interval + val_data.hist + val_data.lag,
    val_data.interval,
)
assert np.nansum(val_data.outputs.isel(time=ind_out).to_array().to_numpy() - val_data.outputs.isel(time=xr.Variable(["window_dim", "time"], val_data.rolling_indices.isel(window_dim=idx))).isel(time=-1).to_array().to_numpy()) == 0.0

In [307]:
assert (d1[0][2][:77] == d1[1][1]).all()
assert (d0[1] == d1[1][0]).all()

In [308]:
hist = 1
val_data = data_CNN_Disk(
    data,
    inputs,
    extra_in,
    outputs,
    wet,
    data_mean,
    data_std,
    N_val,
    lag,
    interval,
    hist,
    e_train,
    device="cuda",
)

/global/common/software/m4331/suryad/conda/emulator/lib/python3.10/site-packages/xarray/core/duck_array_ops.py:188: RuntimeWarning: invalid value encountered in cast
  return data.astype(dtype, **kwargs)


In [309]:
%%time
d2 = val_data[6:10]
print(d2[0].shape)
print(d2[1].shape)

torch.Size([4, 157, 180, 360])
torch.Size([4, 77, 180, 360])
CPU times: user 6.86 s, sys: 1.97 s, total: 8.82 s
Wall time: 4.56 s


In [310]:
assert (d2[1][0] == d1[1][1]).all()

#### data_CNN_Disk_steps

##### old

In [332]:
class old_data_CNN_Disk_steps(torch.utils.data.Dataset):

    def __init__(
        self,
        data,
        inputs_str,
        extra_in_str,
        outputs_str,
        wet,
        data_mean,
        data_std,
        n_samples,
        lag,
        interval,
        steps,
        device="cuda",
    ):
        super().__init__()
        self.device = device

        self.size = n_samples
        self.lag = lag
        self.interval = interval
        self.steps = steps
        
        self.inputs = data[inputs_str + extra_in_str]
        self.outputs = data[outputs_str]

        self.in_mean = data_mean[inputs_str + extra_in_str]
        self.in_std = data_std[inputs_str + extra_in_str]

        self.out_mean = data_mean[outputs_str]
        self.out_std = data_std[outputs_str]

        self.wet = wet

    def set_device(self, device):
        self.device = device

    def __len__(self):
        # Number of data point we have. Alternatively self.data.shape[0], or self.label.shape[0]
        return self.size

    def __getitem__(self, idx):
        # Return the idx-th data point of the dataset
        # If we have multiple things to return (data point and label), we can return them as tuple
        outputs = []
        for step in range(self.steps):
            if type(idx) == list:
                ind_in = [i * self.interval + self.lag * step for i in idx]
                ind_out = [i * self.interval + self.lag * (step + 1) for i in idx]

            elif type(idx) == slice:
                if idx.start == None and idx.stop == None:
                    idx = slice(0, self.size, idx.step)
                elif idx.start == None:
                    idx = slice(0, idx.stop, idx.step)
                elif idx.stop == None:
                    idx = slice(idx.start, self.size, idx.step)

                ind_in = slice(
                    idx.start, idx.stop * self.interval + self.lag * step, self.interval
                )
                ind_out = slice(
                    idx.start + self.lag,
                    idx.stop * self.interval + self.lag * (step + 1),
                    self.interval,
                )

            if type(idx) == int:
                ind_in = idx * self.interval + self.lag * step
                ind_out = idx * self.interval + self.lag * (step + 1)

            data_in = self.inputs.isel(time=ind_in)
            data_in = ((data_in - self.in_mean) / self.in_std).fillna(0)
            label = self.outputs.isel(time=ind_out)
            label = ((label - self.out_mean) / self.out_std).fillna(0)

            if type(idx) == int:
                data_in = data_in.to_array().transpose("variable", "y", "x").to_numpy()
                label = label.to_array().transpose("variable", "y", "x").to_numpy()
            else:
                data_in = (
                    data_in.to_array()
                    .transpose("time", "variable", "y", "x")
                    .to_numpy()
                )
                label = (
                    label.to_array().transpose("time", "variable", "y", "x").to_numpy()
                )

            outputs.append(torch.from_numpy(data_in).float())
            outputs.append(torch.from_numpy(label).float())

        return outputs

##### new

In [353]:
class data_CNN_Disk_steps(torch.utils.data.Dataset):
    
    def __init__(
        self,
        data,
        inputs_str,
        extra_in_str,
        outputs_str,
        wet,
        data_mean,
        data_std,
        n_samples,
        lag,
        interval,
        hist,
        steps,
        device="cuda",
    ):
        super().__init__()
        self.device = device

        self.size = n_samples
        self.lag = lag
        self.interval = interval
        self.hist = hist
        self.steps = steps
        
        assert self.interval == 1
        assert self.lag == 1
        
        self.inputs = data[inputs_str + extra_in_str]
        self.outputs = data[outputs_str]
        
        total_steps = self.hist + 1
        
        indices = xr.DataArray(
            np.arange(data.time.size),
            dims=["time"],
            coords={"time": data.time},
        )
        
        rolling_indices = indices.rolling(time=len(data.time)-total_steps, center=False).construct("window_dim").astype(int)
        self.rolling_indices = rolling_indices.transpose('window_dim', 'time').isel(time=slice(len(data.time)-total_steps-1, None))
        self.inputs_no_extra = data[inputs_str]
        self.extras = data[extra_in_str]
        self.inputs_no_extra_mean = data_mean[inputs_str]
        self.inputs_no_extra_std = data_std[inputs_str]
        self.extras_mean = data_mean[extra_in_str]
        self.extras_std = data_std[extra_in_str]
        
        self.in_mean = data_mean[inputs_str + extra_in_str]
        self.in_std = data_std[inputs_str + extra_in_str]

        self.out_mean = data_mean[outputs_str]
        self.out_std = data_std[outputs_str]

        self.wet = wet

    def set_device(self, device):
        self.device = device

    def __len__(self):
        # Number of data point we have. Alternatively self.data.shape[0], or self.label.shape[0]
        return self.rolling_indices.window_dim.size - self.steps
    
    def __getitem__(self, idx):
        # Return the idx-th data point of the dataset
        # If we have multiple things to return (data point and label), we can return them as tuple
        # Return the idx-th data point of the dataset
        # If we have multiple things to return (data point and label), we can return them as tuple
        outputs = []
        
        assert type(idx) == int
            
        for step in range(self.steps): 
            
            start = idx + step
            end = idx + step + 1
            idx_slice = slice(
                    start, end, self.interval
            )
            
            data_in = self.inputs_no_extra.isel(time=xr.Variable(["window_dim", "time"], self.rolling_indices.isel(window_dim=idx_slice))).isel(time=slice(None, -1))
            data_in = ((data_in - self.inputs_no_extra_mean) / self.inputs_no_extra_std).fillna(0)
            data_in = data_in.to_array().transpose("window_dim", "time", "variable", "y", "x").to_numpy()
            data_in = data_in.reshape((data_in.shape[0], -1, data_in.shape[3], data_in.shape[4]))
            data_in_boundary = self.extras.isel(time=xr.Variable(["window_dim", "time"], self.rolling_indices.isel(window_dim=idx_slice))).isel(time=-2)
            data_in_boundary = ((data_in_boundary - self.extras_mean) / self.extras_std).fillna(0)
            data_in_boundary = data_in_boundary.to_array().transpose("window_dim", "variable", "y", "x").to_numpy()
            data_in = np.concatenate((data_in, data_in_boundary), axis=1).squeeze()

            label = self.outputs.isel(time=xr.Variable(["window_dim", "time"], self.rolling_indices.isel(window_dim=idx_slice))).isel(time=-1)
            label = ((label - self.out_mean) / self.out_std).fillna(0)
            label = label.to_array().transpose("window_dim", "variable", "y", "x").to_numpy().squeeze()

            
            outputs.append(torch.from_numpy(data_in).float())
            outputs.append(torch.from_numpy(label).float())

        return outputs

In [379]:
train_data.rolling_indices.window_dim.size

4743

In [354]:
train_data = old_data_CNN_Disk_steps(
    data,
    inputs,
    extra_in,
    outputs,
    wet,
    data_mean,
    data_std,
    N_samples,
    lag,
    interval,
    steps,
    device="cuda",
)

d0 = train_data[2]
print(d0[0].shape)
print(d0[1].shape)

torch.Size([80, 180, 360])
torch.Size([77, 180, 360])


In [355]:
hist = 0
train_data = data_CNN_Disk_steps(
    data,
    inputs,
    extra_in,
    outputs,
    wet,
    data_mean,
    data_std,
    N_samples,
    lag,
    interval,
    hist,
    steps,
    device="cuda",
)

d = train_data[2]
print(d[0].shape)
print(d[1].shape)

/global/common/software/m4331/suryad/conda/emulator/lib/python3.10/site-packages/xarray/core/duck_array_ops.py:188: RuntimeWarning: invalid value encountered in cast
  return data.astype(dtype, **kwargs)


torch.Size([80, 180, 360])
torch.Size([77, 180, 360])


In [356]:
assert (d[0] == d0[0]).all()
assert (d[1] == d0[1]).all()

In [371]:
for i in range(len(d0)):
    if i % 2 == 0:
        assert (d0[i][:77] == d[i][:77]).all()
    else:
        assert (d0[i] == d[i]).all()

In [357]:
hist = 1
train_data = data_CNN_Disk_steps(
    data,
    inputs,
    extra_in,
    outputs,
    wet,
    data_mean,
    data_std,
    N_samples,
    lag,
    interval,
    hist,
    steps,
    device="cuda",
)


/global/common/software/m4331/suryad/conda/emulator/lib/python3.10/site-packages/xarray/core/duck_array_ops.py:188: RuntimeWarning: invalid value encountered in cast
  return data.astype(dtype, **kwargs)


In [361]:
d2 = train_data[2]
print(d2[0].shape)
print(d2[1].shape)

torch.Size([157, 180, 360])
torch.Size([77, 180, 360])


In [375]:
# [0], [1], [1], [2] ---- [0, 1], [2], [1, 2], [3]


assert (d0[0][:77] == d2[0][:77]).all()
assert (d0[1] == d2[0][77:154]).all()
assert (d0[3] == d2[1]).all()


### Testing

In [57]:
# Create xarray dataset of size - (50, 60, 30, 30)
data = xr.DataArray(
    np.random.rand(5, 2, 1, 1),
    dims=["time", "variable", "y", "x"],
    coords={
        "time": np.arange(5),
        "variable": np.arange(2),
        "y": np.arange(1),
        "x": np.arange(1),
    },
)

data_without_boundary = data.isel(variable=slice(0, 5))
boundary = data.isel(variable=slice(5, 2))



In [58]:
# Create rolling indices of timesteps 
indices = xr.DataArray(
    np.arange(data.time.size),
    dims=["time"],
    coords={"time": data.time},
)
total_steps = hist + 1
# convert to long format
rolling_indices = indices.rolling(time=len(data.time)-total_steps, center=False).construct("window_dim").astype(int)
rolling_indices = rolling_indices.transpose('window_dim', 'time').isel(time=slice(len(data.time)-total_steps-1, None))
rolling_indices
                                    

/global/common/software/m4331/suryad/conda/emulator/lib/python3.10/site-packages/xarray/core/duck_array_ops.py:188: RuntimeWarning: invalid value encountered in cast
  return data.astype(dtype, **kwargs)


<xarray.DataArray (window_dim: 3, time: 3)>
array([[0, 1, 2],
       [1, 2, 3],
       [2, 3, 4]])
Coordinates:
  * time     (time) int64 2 3 4
Dimensions without coordinates: window_dim

In [59]:
# Create rolling dataset itself
rolling_data = data.rolling(time=len(data.time)-total_steps, center=False).construct('window_dim')
data2 = rolling_data.transpose('window_dim', 'time', 'variable', 'y', 'x').isel(time=slice(len(data.time)-total_steps-1,None))

In [60]:
data2.isel(window_dim=2)

<xarray.DataArray (time: 3, variable: 2, y: 1, x: 1)>
array([[[[0.25514849]],

        [[0.14194677]]],


       [[[0.48343975]],

        [[0.54593093]]],


       [[[0.90935109]],

        [[0.93858622]]]])
Coordinates:
  * time      (time) int64 2 3 4
  * variable  (variable) int64 0 1
  * y         (y) int64 0
  * x         (x) int64 0

In [61]:
data.isel(time=slice(2, total_steps+1+2))

<xarray.DataArray (time: 3, variable: 2, y: 1, x: 1)>
array([[[[0.25514849]],

        [[0.14194677]]],


       [[[0.48343975]],

        [[0.54593093]]],


       [[[0.90935109]],

        [[0.93858622]]]])
Coordinates:
  * time      (time) int64 2 3 4
  * variable  (variable) int64 0 1
  * y         (y) int64 0
  * x         (x) int64 0

In [62]:
np.equal(data2.isel(window_dim=2), data.isel(time=slice(2, total_steps+1+2))).all()

<xarray.DataArray ()>
array(True)

In [63]:
# %%timeit
def slice_indices(size, slices):
    range_ = range(size)
    return np.array([range_[slice_] for slice_ in slices])

slices = [slice(0, 2), slice(1, 3)]
dims = ["window_dim", "time"]
print(xr.Variable(dims, slice_indices(data.sizes["time"], slices)))
d = data.isel(time=xr.Variable(dims, slice_indices(data.sizes["time"], slices)))
d.load()


<xarray.Variable (window_dim: 2, time: 2)>
array([[0, 1],
       [1, 2]])


<xarray.DataArray (window_dim: 2, time: 2, variable: 2, y: 1, x: 1)>
array([[[[[0.36373818]],

         [[0.12655502]]],


        [[[0.74645363]],

         [[0.99822165]]]],



       [[[[0.74645363]],

         [[0.99822165]]],


        [[[0.25514849]],

         [[0.14194677]]]]])
Coordinates:
    time      (window_dim, time) int64 0 1 1 2
  * variable  (variable) int64 0 1
  * y         (y) int64 0
  * x         (x) int64 0
Dimensions without coordinates: window_dim

In [65]:
# Use rolling indices to get the rolling dataset
dims = ["window_dim", "time"]
data3 = data.isel(time=xr.Variable(dims, rolling_indices))
data3

<xarray.DataArray (window_dim: 3, time: 3, variable: 2, y: 1, x: 1)>
array([[[[[0.36373818]],

         [[0.12655502]]],


        [[[0.74645363]],

         [[0.99822165]]],


        [[[0.25514849]],

         [[0.14194677]]]],



       [[[[0.74645363]],

         [[0.99822165]]],

...

        [[[0.48343975]],

         [[0.54593093]]]],



       [[[[0.25514849]],

         [[0.14194677]]],


        [[[0.48343975]],

         [[0.54593093]]],


        [[[0.90935109]],

         [[0.93858622]]]]])
Coordinates:
    time      (window_dim, time) int64 0 1 2 1 2 3 2 3 4
  * variable  (variable) int64 0 1
  * y         (y) int64 0
  * x         (x) int64 0
Dimensions without coordinates: window_dim

In [35]:
np.equal(data3.isel(window_dim=10), data2.isel(window_dim=10)).all()

<xarray.DataArray ()>
array(True)

In [438]:

def forward_once(input_tensor):
    return input_tensor[:, :output_channels]

def forward(
        inputs,
        output_only_last=False,
        loss_fn=None,
    ) -> torch.Tensor:
        # HIST=0, 4 steps ; 0->[0in, 1out; 1in, 2out; 2in, 3out; 3in, 4out]
        # HIST=1, 4 steps ; 0->[[0in, 1in], 2out; [1in, 2in], 3out; [2in, 3in], 4out; [3in, 4in], 5out]
        # HIST=2, 4 steps ; 0->[[0in, 1in, 2in], 3out; [1in, 2in, 3in], 4out; [2in, 3in, 4in], 5out; [3in, 4in, 5in], 6out]
        outputs = []
        loss = None
        ### TEMP
        for n in range(len(inputs)):
            inputs[n] = inputs[n].unsqueeze(0)
        
        N, C, H, W = inputs[0].shape

        for step in range(len(inputs) // 2):
            print(step)
            if step == 0:
                input_tensor = inputs[0]
            elif step <= hist:
                inputs_0 = inputs[2*step][:, :output_channels*(hist-step+1)]
                inputs_1 = torch.cat([outputs[i] for i in range(step)], dim=1)
                print("inputs_0", inputs_0.shape, "slice: ", None, output_channels*(hist-step+1))
                print("inputs_1", inputs_1.shape)
                print("inputs[2*step][0][:, output_channels*(hist+1) :]", inputs[2*step][:, output_channels*(hist+1) :].shape, "Slice: ", output_channels*(hist+1), None)
                input_tensor = torch.cat(
                    [
                        inputs_0,
                        inputs_1,
                        inputs[2*step][:, output_channels*(hist+1) :],
                    ],
                    dim=1,
                )   
            else:
                inputs_0 = torch.cat([outputs[i] for i in range(-hist-1, 0)], dim=1)
                print("inputs_0", inputs_0.shape, "outputs[]", [i for i in range(-hist-1, 0)])
                print("inputs[2*step]", inputs[2*step].shape, "Slice: ", output_channels*(hist+1), None)
                input_tensor = torch.cat(
                    [
                        inputs_0,
                        inputs[2*step][:, output_channels*(hist+1) :],
                    ],
                    dim=1,
                )
            print(input_tensor.shape)
            
            assert input_tensor.shape[1] == input_channels, f"Input shape is {input_tensor.shape[1]} but should be {input_channels}"
            decodings = forward_once(input_tensor)
            if pred_residuals:
                reshaped = (
                    input_tensor[:, output_channels*hist : output_channels*(hist+1)] + decodings
                )  # Residual prediction
            else:
                reshaped = decodings  # Absolute prediction

            if loss_fn is not None:
                if loss is None:
                    loss = loss_fn(
                        reshaped,
                        inputs[2 * step + 1],
                    )
                else:
                    loss += loss_fn(
                        reshaped,
                        inputs[2 * step + 1],
                    )

            outputs.append(reshaped)

        if loss_fn is None:
            if output_only_last:
                res = outputs[-1]
            else:
                res = outputs
            return res

        else:
            return loss

In [441]:
hist = 0
output_channels = 77
input_channels = (hist+1)*77+3
pred_residuals = False

train_data = data_CNN_Disk_steps(
    data,
    inputs,
    extra_in,
    outputs,
    wet,
    data_mean,
    data_std,
    N_samples,
    lag,
    interval,
    hist,
    steps,
    device="cuda",
)
a = forward(train_data[2], output_only_last=False, loss_fn=torch.nn.MSELoss())

/global/common/software/m4331/suryad/conda/emulator/lib/python3.10/site-packages/xarray/core/duck_array_ops.py:188: RuntimeWarning: invalid value encountered in cast
  return data.astype(dtype, **kwargs)


0
torch.Size([1, 80, 180, 360])
1
inputs_0 torch.Size([1, 77, 180, 360]) outputs[] [-1]
inputs[2*step] torch.Size([1, 80, 180, 360]) Slice:  77 None
torch.Size([1, 80, 180, 360])
2
inputs_0 torch.Size([1, 77, 180, 360]) outputs[] [-1]
inputs[2*step] torch.Size([1, 80, 180, 360]) Slice:  77 None
torch.Size([1, 80, 180, 360])
3
inputs_0 torch.Size([1, 77, 180, 360]) outputs[] [-1]
inputs[2*step] torch.Size([1, 80, 180, 360]) Slice:  77 None
torch.Size([1, 80, 180, 360])


In [421]:
def inference(
        inputs, hist, num_steps=None, output_only_last=False, device="cuda"
    ) -> torch.Tensor:
        # HIST=0 ; 0->[0, 1]; 1->[1, 2]; 2->[2, 3]; 3->[3, 4]
        # HIST=1 ; 0->[[0, 1], 2]; 1->[[1, 2], 3]; 2->[[2, 3], 4]; 3->[[3, 4], 5]
        # HIST=2 ; 0->[[0, 1, 2], 3]; 1->[[1, 2, 3], 4]; 2->[[2, 3, 4], 5]; 3->[[3, 4, 5], 6]
        outputs = []
        
        for step in range(num_steps):
            print(step)
            if step == 0:
                input_tensor = inputs[0][0].to(device=device)
                print(input_tensor.shape)
            elif step <= hist:

                inputs_0 = inputs[step][0][0, :output_channels*(hist-step+1)].unsqueeze(0)
                inputs_1 = torch.cat([outputs[i].unsqueeze(0) for i in range(step)], dim=1)
                print("inputs_0", inputs_0.shape, "slice: ", None, output_channels*(hist-step+1))
                print("inputs_1", inputs_1.shape)
                print("inputs[step][0][0, output_channels*(hist+1) :].unsqueeze(0)", inputs[step][0][0, output_channels*(hist+1) :].unsqueeze(0).shape, "Slice: ", output_channels*(hist+1), None)
                input_tensor = torch.cat(
                    [
                        inputs_0,
                        inputs_1,
                        inputs[step][0][0, output_channels*(hist+1) :].unsqueeze(0)
                        .to(device=device),
                    ],
                    dim=1,
                )   
                print(input_tensor.shape)
            else:
                inputs_0 = torch.cat([outputs[i].unsqueeze(0) for i in range(-hist-1, 0)], dim=1)
                print("inputs_0", inputs_0.shape, "outputs[]", [i for i in range(-hist-1, 0)])
                print("inputs[step][0]", inputs[step][0].shape, "Slice: ", output_channels*(hist+1), None)
                input_tensor = torch.cat(
                    [
                        inputs_0,
                        inputs[step][0][0, output_channels*(hist+1) :].unsqueeze(0)
                        .to(device=device),
                    ],
                    dim=1,
                )
                print(input_tensor.shape)
            
            assert input_tensor.shape[1] == input_channels, f"Input shape is {input_tensor.shape[1]} but should be {input_channels}"
            decodings = forward_once(input_tensor)
            if pred_residuals:
                reshaped = input_tensor[0, output_channels*hist : output_channels*(hist+1)].to(
                    device=device
                ) + decodings.squeeze(
                    0
                )
            else:
                reshaped = decodings.squeeze(0)
            
            outputs.append(reshaped)

        if output_only_last:
            res = outputs[-1]
        else:
            res = outputs

        return res


In [422]:
hist = 0
output_channels = 77
input_channels = (hist+1)*77+3
pred_residuals = False

val_data = data_CNN_Disk(
    data,
    inputs,
    extra_in,
    outputs,
    wet,
    data_mean,
    data_std,
    N_val,
    lag,
    interval,
    hist,
    e_train,
    device="cuda",
)
inf = inference(val_data, hist, num_steps=10, output_only_last=False, device="cpu")

/global/common/software/m4331/suryad/conda/emulator/lib/python3.10/site-packages/xarray/core/duck_array_ops.py:188: RuntimeWarning: invalid value encountered in cast
  return data.astype(dtype, **kwargs)


0
torch.Size([1, 80, 180, 360])
1
inputs_0 torch.Size([1, 77, 180, 360]) outputs[] [-1]
inputs[step][0] torch.Size([1, 80, 180, 360]) Slice:  77 None
torch.Size([1, 80, 180, 360])
2
inputs_0 torch.Size([1, 77, 180, 360]) outputs[] [-1]
inputs[step][0] torch.Size([1, 80, 180, 360]) Slice:  77 None
torch.Size([1, 80, 180, 360])
3
inputs_0 torch.Size([1, 77, 180, 360]) outputs[] [-1]
inputs[step][0] torch.Size([1, 80, 180, 360]) Slice:  77 None
torch.Size([1, 80, 180, 360])
4
inputs_0 torch.Size([1, 77, 180, 360]) outputs[] [-1]
inputs[step][0] torch.Size([1, 80, 180, 360]) Slice:  77 None
torch.Size([1, 80, 180, 360])
5
inputs_0 torch.Size([1, 77, 180, 360]) outputs[] [-1]
inputs[step][0] torch.Size([1, 80, 180, 360]) Slice:  77 None
torch.Size([1, 80, 180, 360])
6
inputs_0 torch.Size([1, 77, 180, 360]) outputs[] [-1]
inputs[step][0] torch.Size([1, 80, 180, 360]) Slice:  77 None
torch.Size([1, 80, 180, 360])
7
inputs_0 torch.Size([1, 77, 180, 360]) outputs[] [-1]


KeyboardInterrupt: 